# SSD故障日志深度分析与预测建模

本 notebook 记录深度分析部分的完整流程，包括：

- SMART 指标分布分析
- 多指标相关性与健康/故障盘对比
- 位置与业务维度的关联故障分析
- 基于 SMART、型号、应用和槽位信息的故障预测建模

为了避免跨型号误匹配，本文以 `(model, disk_id)` 作为样本主键。


In [1]:
# 单元格 1：导入库与路径设置
# 说明：
# 1. 代码不直接写死中文目录名，而是自动寻找 DataAnalyse.ipynb 和 dcbrain/ssd_open_data。
# 2. 所有图表输出到 源代码/figures，统计结果输出到 源代码/results。

from pathlib import Path
import json

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")

cwd = Path.cwd()
if (cwd / "DataAnalyse.ipynb").exists() or (cwd / "MyAnalysis.ipynb").exists():
    SRC_DIR = cwd
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd
    matches = list(PROJECT_ROOT.glob("*/DataAnalyse.ipynb"))
    if not matches:
        matches = list(PROJECT_ROOT.glob("*/MyAnalysis.ipynb"))
    if not matches:
        raise FileNotFoundError("Cannot locate notebook directory from current working directory.")
    SRC_DIR = matches[0].parent

FIG_DIR = SRC_DIR / "figures"
RESULT_DIR = SRC_DIR / "results"
FIG_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

data_roots = list(PROJECT_ROOT.glob("*/dcbrain/ssd_open_data"))
if not data_roots:
    raise FileNotFoundError("Cannot locate dcbrain/ssd_open_data.")
DATA_ROOT = data_roots[0]

SMART_PATH = DATA_ROOT / "smart_log_20191231.csv" / "20191231.csv"
FAILURE_PATH = DATA_ROOT / "ssd_failure_tag.csv" / "ssd_failure_tag.csv"
LOCATION_PATH = DATA_ROOT / "location_info_of_ssd.csv" / "location_info_of_ssd.csv"

FEATURES = ["r_5", "r_9", "r_12", "r_175", "n_5", "n_9", "n_12", "n_175"]
RAW_FEATURES = ["r_5", "r_9", "r_12", "r_175"]

print("SRC_DIR:", SRC_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("SMART exists:", SMART_PATH.exists())
print("failure tag exists:", FAILURE_PATH.exists())
print("location info exists:", LOCATION_PATH.exists())


SRC_DIR: D:\大数据处理\大作业\源代码
DATA_ROOT: D:\大数据处理\大作业\数据集\dcbrain\ssd_open_data
SMART exists: True
failure tag exists: True
location info exists: True


## 1. 数据读取与主键构建

原始 SMART 日志、故障标签和位置信息分别来自三个 CSV 文件。由于同一个 `disk_id` 可能出现在不同型号中，因此这里使用 `(model, disk_id)` 作为主键。


In [2]:
# 单元格 2：读取 SMART、故障标签和位置数据

smart = pd.read_csv(
    SMART_PATH,
    usecols=["disk_id", "ds", "model", *FEATURES],
)

failure_tag = pd.read_csv(
    FAILURE_PATH,
    usecols=["model", "disk_id", "failure"],
).drop_duplicates(["model", "disk_id"])

location_info = pd.read_csv(
    LOCATION_PATH,
    usecols=["model", "disk_id", "app", "rack_id", "node_id", "slot_id"],
).drop_duplicates(["model", "disk_id"])

print("SMART rows:", len(smart))
print("SMART unique (model, disk_id):", smart.drop_duplicates(["model", "disk_id"]).shape[0])
print("failure tag rows:", len(failure_tag))
print("location rows:", len(location_info))


SMART rows: 706182
SMART unique (model, disk_id): 706182
failure tag rows: 18387
location rows: 965495


In [3]:
# 单元格 3：合并数据并构造位置特征
# 关键点：
# 1. SMART 与 failure_tag 按 (model, disk_id) 合并。
# 2. 未匹配到 failure 标签的样本视为健康盘。
# 3. slot_id 缺失值保留为 missing；样本数过少的槽位合并为 rare_slot。

data = smart.merge(failure_tag, on=["model", "disk_id"], how="left")
data["failure"] = data["failure"].fillna(0).astype(int)

data = data.merge(location_info, on=["model", "disk_id"], how="left", indicator="location_merge")
location_match_rate = (data["location_merge"] == "both").mean()
data = data.drop(columns=["location_merge"])

data["app"] = data["app"].fillna("unknown")
data["slot_id_cat"] = data["slot_id"].where(data["slot_id"].notna(), "missing").astype(str)

slot_counts = data["slot_id_cat"].value_counts()
common_slots = set(slot_counts[slot_counts >= 100].index)
data["slot_id_grouped"] = data["slot_id_cat"].where(data["slot_id_cat"].isin(common_slots), "rare_slot")

data.to_csv(SRC_DIR / "my_analysis_dataset.csv", index=False)

print("merged rows:", len(data))
print("unique (model, disk_id):", data.drop_duplicates(["model", "disk_id"]).shape[0])
print("failure counts:")
print(data["failure"].value_counts().sort_index())
print("location match rate:", f"{location_match_rate:.2%}")
print("slot_id missing rate:", f"{data['slot_id'].isna().mean():.2%}")


merged rows: 706182
unique (model, disk_id): 706182
failure counts:
failure
0    702723
1      3459
Name: count, dtype: int64
location match rate: 100.00%
slot_id missing rate: 18.02%


## 2. SMART 指标分析

这一部分对多个 SMART 指标进行健康盘与故障盘对比，观察哪些指标在故障盘中存在明显差异。


In [4]:
# 单元格 4：其他 SMART 指标箱线图
# 使用 log1p 缩放，避免极端大值影响图形可读性。

plot_data = data[["failure"] + RAW_FEATURES].copy()
for col in RAW_FEATURES:
    plot_data[col] = np.log1p(plot_data[col].clip(lower=0))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, col in zip(axes, ["r_9", "r_12", "r_175"]):
    sns.boxplot(
        data=plot_data,
        x="failure",
        y=col,
        hue="failure",
        palette={0: "#4C78A8", 1: "#F58518"},
        showfliers=False,
        legend=False,
        ax=ax,
    )
    ax.set_title(f"{col} distribution by failure")
    ax.set_xlabel("failure (0=healthy, 1=failed)")
    ax.set_ylabel(f"log1p({col})")

fig.suptitle("Other SMART Raw Indicators", fontsize=14)
fig.savefig(FIG_DIR / "smart_other_boxplots.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\admin\AppData\Local\Temp\ipykernel_44372\911197812.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# 单元格 5：健康盘与故障盘的 SMART 指标均值对比

indicator_rows = []
for feature in FEATURES:
    healthy = data.loc[data["failure"] == 0, feature]
    failed = data.loc[data["failure"] == 1, feature]
    indicator_rows.append(
        {
            "feature": feature,
            "healthy_mean": healthy.mean(),
            "failed_mean": failed.mean(),
            "failed_to_healthy_mean_ratio": failed.mean() / healthy.mean() if healthy.mean() != 0 else np.nan,
            "healthy_median": healthy.median(),
            "failed_median": failed.median(),
            "missing_pct": data[feature].isna().mean() * 100,
        }
    )

indicator_summary = pd.DataFrame(indicator_rows)
indicator_summary.to_csv(RESULT_DIR / "indicator_failure_comparison.csv", index=False)
indicator_summary


,feature,healthy_mean,failed_mean,failed_to_healthy_mean_ratio,healthy_median,failed_median,missing_pct
0,r_5,7.061879e+00,1.264047e+02,17.899589,0.000000e+00,3.000000e+00,0.000142
1,r_9,3.044016e+04,2.788430e+04,0.916037,3.099700e+04,2.322100e+04,0.000142
2,r_12,3.181004e+01,3.557865e+01,1.118472,2.000000e+01,2.400000e+01,0.906140
3,r_175,9.780950e+11,1.121985e+12,1.147113,9.841029e+11,1.215755e+12,50.798519
4,n_5,9.972964e+01,9.810292e+01,0.983689,1.000000e+02,1.000000e+02,0.000142
5,n_9,9.848883e+01,9.889072e+01,1.004081,1.000000e+02,1.000000e+02,0.000142
6,n_12,9.975370e+01,9.978060e+01,1.000270,1.000000e+02,1.000000e+02,0.906140
7,n_175,9.984652e+01,8.813483e+01,0.882703,1.000000e+02,1.000000e+02,50.798519


In [6]:
# 单元格 6：SMART 指标相关性热力图

corr_data = data[FEATURES + ["failure"]].copy()
for col in RAW_FEATURES:
    corr_data[col] = np.log1p(corr_data[col].clip(lower=0))

corr = corr_data.corr(numeric_only=True)
corr.to_csv(RESULT_DIR / "smart_correlation_matrix.csv")

plt.figure(figsize=(8, 6))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f", linewidths=0.5)
plt.title("SMART Feature Correlation Heatmap")
plt.savefig(FIG_DIR / "smart_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\admin\AppData\Local\Temp\ipykernel_44372\2406291844.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# 单元格 7：故障盘与健康盘的多指标小提琴图
# 健康样本远多于故障样本，因此绘图时对每类最多抽样 8000 条。

samples = []
for label, group in data.groupby("failure"):
    samples.append(group.sample(n=min(8000, len(group)), random_state=42))
sample_df = pd.concat(samples, ignore_index=True)

long_df = sample_df.melt(
    id_vars=["failure"],
    value_vars=FEATURES,
    var_name="feature",
    value_name="value",
)
long_df["log_value"] = np.log1p(long_df["value"].clip(lower=0))

plt.figure(figsize=(13, 6))
sns.violinplot(
    data=long_df,
    x="feature",
    y="log_value",
    hue="failure",
    split=True,
    inner="quartile",
    palette={0: "#4C78A8", 1: "#F58518"},
    cut=0,
)
plt.title("Healthy vs Failed Disks: Multi-feature Distribution")
plt.xlabel("SMART feature")
plt.ylabel("log1p(value)")
plt.savefig(FIG_DIR / "smart_multi_metric_violin.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\admin\AppData\Local\Temp\ipykernel_44372\244815048.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# 单元格 8：按型号统计故障率与关键指标分布

model_stats = (
    data.groupby("model")
    .agg(
        sample_count=("disk_id", "count"),
        failed_count=("failure", "sum"),
        failure_rate=("failure", "mean"),
        r_5_median=("r_5", "median"),
        r_9_median=("r_9", "median"),
        r_12_median=("r_12", "median"),
        r_175_median=("r_175", "median"),
    )
    .reset_index()
    .sort_values("failure_rate", ascending=False)
)
model_stats.to_csv(RESULT_DIR / "model_failure_feature_summary.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
sns.barplot(data=model_stats, x="model", y="failure_rate", hue="model", palette="viridis", legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Failure rate by model")

for ax, feature in zip(axes.ravel()[1:], ["r_5", "r_9", "r_12"]):
    temp = data[["model", feature]].copy()
    temp[feature] = np.log1p(temp[feature].clip(lower=0))
    sns.boxplot(data=temp, x="model", y=feature, showfliers=False, color="#72B7B2", ax=ax)
    ax.set_title(f"{feature} distribution by model")
    ax.set_ylabel(f"log1p({feature})")

fig.savefig(FIG_DIR / "model_failure_and_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

model_stats.head(10)


C:\Users\admin\AppData\Local\Temp\ipykernel_44372\2014642992.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,model,sample_count,failed_count,failure_rate,r_5_median,r_9_median,r_12_median,r_175_median
2,A3,16774,617,0.036783,0.0,48366.5,38.0,1.215984e+12
8,B3,38564,562,0.014573,0.0,21119.5,19.0,NaN
9,C1,167879,1487,0.008858,0.0,18350.0,25.0,NaN
10,C2,19776,162,0.008192,0.0,12300.0,23.0,NaN
5,A6,12824,67,0.005225,0.0,39694.0,39.0,1.005326e+12
1,A2,54816,118,0.002153,0.0,38676.0,17.0,9.879085e+11
7,B2,43443,86,0.001980,0.0,29312.0,24.0,NaN
3,A4,35722,66,0.001848,0.0,28850.5,28.0,7.306805e+11
4,A5,27863,31,0.001113,0.0,28181.0,20.0,7.219449e+11
6,B1,89067,94,0.001055,0.0,32533.0,17.0,NaN


## 3. 位置与业务维度关联故障分析

这一部分使用 `location_info` 中的应用、机架和槽位信息，观察故障是否存在业务或空间聚集现象。


In [9]:
# 单元格 9：定义分组故障率统计函数

global_rate = data["failure"].mean()

def summarize_group(df, col, min_n=1):
    stats = (
        df.groupby(col, dropna=False)
        .agg(sample_count=("failure", "size"), failed_count=("failure", "sum"), failure_rate=("failure", "mean"))
        .reset_index()
    )
    stats["expected_failed_at_global_rate"] = stats["sample_count"] * global_rate
    stats["lift_vs_global"] = stats["failure_rate"] / global_rate
    stats = stats[stats["sample_count"] >= min_n].sort_values(["failure_rate", "failed_count"], ascending=False)
    return stats


In [10]:
# 单元格 10：不同应用的故障率

app_stats = summarize_group(data, "app")
app_stats.to_csv(RESULT_DIR / "app_failure_summary.csv", index=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=app_stats, x="app", y="failure_rate", hue="app", palette="crest", legend=False)
plt.axhline(global_rate, color="#D62728", linestyle="--", linewidth=1.4, label="global rate")
plt.title("Failure Rate by Application")
plt.xlabel("application")
plt.ylabel("failure rate")
plt.legend()
plt.savefig(FIG_DIR / "location_app_failure_rate.png", dpi=300, bbox_inches="tight")
plt.show()

app_stats.head(10)


C:\Users\admin\AppData\Local\Temp\ipykernel_44372\1055965549.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,app,sample_count,failed_count,failure_rate,expected_failed_at_global_rate,lift_vs_global
0,DAE,13324,386,0.028970,65.263227,5.914510
2,NAS,12977,259,0.019958,63.563562,4.074662
8,none,182794,1451,0.007938,895.356220,1.620584
1,DB,25393,118,0.004647,124.379249,0.948711
3,RM,82952,370,0.004460,406.313058,0.910628
4,SS,15642,60,0.003836,76.617187,0.783114
7,WSM,329355,752,0.002283,1613.237020,0.466144
5,WPS,34781,52,0.001495,170.363276,0.305230
6,WS,8964,11,0.001227,43.907202,0.250528


In [11]:
# 单元格 11：高故障机架热点
# 机架类别很多，因此只展示样本数不少于 100 的 rack_id，减少小样本误判。

rack_stats = summarize_group(data, "rack_id", min_n=100)
rack_stats.to_csv(RESULT_DIR / "rack_failure_hotspots.csv", index=False)

rack_plot = rack_stats.head(15).copy()
rack_plot["rack_id"] = rack_plot["rack_id"].astype(int).astype(str)

plt.figure(figsize=(12, 6))
sns.barplot(data=rack_plot, y="rack_id", x="failure_rate", hue="rack_id", palette="flare", legend=False)
plt.axvline(global_rate, color="#1F77B4", linestyle="--", linewidth=1.4, label="global rate")
plt.title("Top Rack Failure Hotspots (sample_count >= 100)")
plt.xlabel("failure rate")
plt.ylabel("rack_id")
plt.legend()
plt.savefig(FIG_DIR / "location_rack_hotspots.png", dpi=300, bbox_inches="tight")
plt.show()

rack_stats.head(10)


C:\Users\admin\AppData\Local\Temp\ipykernel_44372\3688467009.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,rack_id,sample_count,failed_count,failure_rate,expected_failed_at_global_rate,lift_vs_global
7552,8544.0,120,63,0.525000,0.587780,107.182871
9923,11248.0,132,65,0.492424,0.646559,100.532274
24717,28062.0,108,46,0.425926,0.529002,86.956121
8204,9289.0,121,33,0.272727,0.592679,55.679413
9653,10937.0,138,36,0.260870,0.675948,53.258569
24124,27387.0,139,36,0.258993,0.680846,52.875414
6321,7144.0,261,62,0.237548,1.278423,48.497267
18232,20661.0,264,48,0.181818,1.293117,37.119609
1964,2216.0,108,16,0.148148,0.529002,30.245607
21914,24850.0,144,17,0.118056,0.705337,24.101968


In [12]:
# 单元格 12：槽位故障率
# slot_id 缺失较多；缺失值保留为 missing，低频槽位合并为 rare_slot。

slot_stats = summarize_group(data, "slot_id_grouped")
slot_stats.to_csv(RESULT_DIR / "slot_failure_summary.csv", index=False)

slot_plot = slot_stats[slot_stats["sample_count"] >= 100].sort_values("failure_rate", ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(
    data=slot_plot,
    x="slot_id_grouped",
    y="failure_rate",
    hue="slot_id_grouped",
    palette="mako",
    legend=False,
)
plt.axhline(global_rate, color="#D62728", linestyle="--", linewidth=1.4, label="global rate")
plt.title("Failure Rate by Slot")
plt.xlabel("slot_id")
plt.ylabel("failure rate")
plt.xticks(rotation=45)
plt.legend()
plt.savefig(FIG_DIR / "location_slot_failure_rate.png", dpi=300, bbox_inches="tight")
plt.show()

slot_stats.head(12)


C:\Users\admin\AppData\Local\Temp\ipykernel_44372\3177780244.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,slot_id_grouped,sample_count,failed_count,failure_rate,expected_failed_at_global_rate,lift_vs_global
16,rare_slot,175,16,0.091429,0.857180,18.665860
1,11.0,289,8,0.027682,1.415571,5.651428
8,22.0,2042,53,0.025955,10.002065,5.298906
15,missing,127240,1247,0.009800,623.243243,2.000824
2,12.0,13120,83,0.006326,64.264000,1.291547
12,27.0,27633,165,0.005971,135.351152,1.219051
3,13.0,32295,190,0.005883,158.186424,1.201114
9,24.0,29485,168,0.005698,144.422564,1.163253
0,0.0,27595,156,0.005653,135.165021,1.154145
14,8.0,27653,150,0.005424,135.449115,1.107427


## 4. 故障预测建模

模型目标不是直接做生产级告警，而是在极端类别不平衡的单日数据上，评估不同特征组合对故障风险排序的帮助。


In [13]:
# 单元格 13：定义模型评估函数

def evaluate_predictions(y_test, y_pred, y_prob):
    cm = confusion_matrix(y_test, y_pred)
    order = np.argsort(y_prob)[::-1]
    y_array = y_test.to_numpy()
    positives = int(y_test.sum())

    top_k = {}
    for k in [100, 500, 1000, 3000, 5000]:
        k_eff = min(k, len(y_array))
        hits = int(y_array[order[:k_eff]].sum())
        top_k[str(k)] = {
            "hits": hits,
            "precision": round(hits / k_eff, 4),
            "recall": round(hits / positives, 4) if positives else 0.0,
        }

    return {
        "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "precision_failed": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
        "recall_failed": round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
        "f1_failed": round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
        "average_precision": round(float(average_precision_score(y_test, y_prob)), 4),
        "confusion_matrix": cm.tolist(),
        "top_k": top_k,
        "classification_report": classification_report(
            y_test,
            y_pred,
            target_names=["healthy", "failed"],
            zero_division=0,
            output_dict=True,
        ),
    }


In [14]:
# 单元格 14：训练三组逻辑回归模型
# Baseline: SMART 数值特征
# Model-aware: SMART + model
# Location-aware: SMART + model + app + slot_id_grouped

model_df = data[FEATURES + ["model", "app", "slot_id_grouped", "failure"]].copy()
X = model_df.drop(columns=["failure"])
y = model_df["failure"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

def build_pipeline(cat_features):
    transformers = [
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            FEATURES,
        )
    ]
    if cat_features:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_features))

    preprocessor = ColumnTransformer(transformers=transformers)

    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=42,
                    solver="lbfgs",
                ),
            ),
        ]
    )

models = {
    "baseline_numeric_logistic": (build_pipeline([]), FEATURES),
    "model_aware_logistic": (build_pipeline(["model"]), FEATURES + ["model"]),
    "location_aware_logistic": (build_pipeline(["model", "app", "slot_id_grouped"]), FEATURES + ["model", "app", "slot_id_grouped"]),
}

model_results = {}
trained_models = {}

for name, (pipeline, cols) in models.items():
    pipeline.fit(X_train[cols], y_train)
    y_pred = pipeline.predict(X_test[cols])
    y_prob = pipeline.predict_proba(X_test[cols])[:, 1]
    model_results[name] = evaluate_predictions(y_test, y_pred, y_prob)
    trained_models[name] = (pipeline, cols)

pd.DataFrame(
    [
        {"model": name, **{k: result[k] for k in ["accuracy", "precision_failed", "recall_failed", "f1_failed", "roc_auc", "average_precision"]}}
        for name, result in model_results.items()
    ]
)


,model,accuracy,precision_failed,recall_failed,f1_failed,roc_auc,average_precision
0,baseline_numeric_logistic,0.6908,0.0130,0.828,0.0256,0.7993,0.0214
1,model_aware_logistic,0.7206,0.0139,0.802,0.0274,0.8208,0.0385
2,location_aware_logistic,0.8065,0.0197,0.789,0.0384,0.8495,0.0385


In [15]:
# 单元格 15：保存模型结果、混淆矩阵和重要特征

best_model, best_cols = trained_models["location_aware_logistic"]
best_result = model_results["location_aware_logistic"]

cm = pd.DataFrame(
    best_result["confusion_matrix"],
    index=["actual_healthy", "actual_failed"],
    columns=["pred_healthy", "pred_failed"],
)
cm.to_csv(RESULT_DIR / "model_confusion_matrix.csv")

cat_names = list(
    best_model.named_steps["preprocess"]
    .named_transformers_["cat"]
    .get_feature_names_out(["model", "app", "slot_id_grouped"])
)
feature_names = FEATURES + cat_names
coefs = best_model.named_steps["model"].coef_[0]
coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefs})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
coef_df.to_csv(RESULT_DIR / "logistic_regression_coefficients.csv", index=False)

summary = {
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "positive_rate_test": round(float(y_test.mean()), 6),
    "features": {
        "baseline": FEATURES,
        "model_aware": FEATURES + ["model"],
        "location_aware": FEATURES + ["model", "app", "slot_id_grouped"],
        "excluded_location_features": ["rack_id", "node_id"],
    },
    "model_results": model_results,
    "top_coefficients": coef_df.head(15).to_dict(orient="records"),
}

with open(RESULT_DIR / "failure_prediction_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Location-aware confusion matrix:")
display(cm)
print("Top coefficients:")
display(coef_df.head(15))


Location-aware confusion matrix:


,pred_healthy,pred_failed
actual_healthy,113366,27179
actual_failed,146,546


Top coefficients:


,feature,coefficient,abs_coefficient
16,model_B3,6.625303,6.625303
15,model_B2,4.115928,4.115928
8,model_A1,-4.078451,4.078451
11,model_A4,-3.966283,3.966283
29,slot_id_grouped_11.0,3.843352,3.843352
14,model_B1,3.699522,3.699522
12,model_A5,-3.640361,3.640361
9,model_A2,-3.533520,3.533520
44,slot_id_grouped_rare_slot,3.161602,3.161602
13,model_A6,-2.538326,2.538326


In [16]:
# 单元格 16：汇总本 notebook 的主要输出

print("Figures saved to:", FIG_DIR)
print("Results saved to:", RESULT_DIR)
print("Key output files:")
for path in [
    FIG_DIR / "smart_other_boxplots.png",
    FIG_DIR / "smart_correlation_heatmap.png",
    FIG_DIR / "smart_multi_metric_violin.png",
    FIG_DIR / "model_failure_and_metrics.png",
    FIG_DIR / "location_app_failure_rate.png",
    FIG_DIR / "location_rack_hotspots.png",
    FIG_DIR / "location_slot_failure_rate.png",
    RESULT_DIR / "failure_prediction_metrics.json",
]:
    print("-", path.name, "exists=", path.exists())


Figures saved to: D:\大数据处理\大作业\源代码\figures
Results saved to: D:\大数据处理\大作业\源代码\results
Key output files:
- smart_other_boxplots.png exists= True
- smart_correlation_heatmap.png exists= True
- smart_multi_metric_violin.png exists= True
- model_failure_and_metrics.png exists= True
- location_app_failure_rate.png exists= True
- location_rack_hotspots.png exists= True
- location_slot_failure_rate.png exists= True
- failure_prediction_metrics.json exists= True
